In [0]:
# Databricks notebook source
# Step 5: Gold layer — dimensional model (dims, facts, SCD Type 2)
# Run in a new notebook: 04_gold_layer

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "fifa_project"

# COMMAND ----------
# --- dim_player ---
dim_player = spark.table(f"{CATALOG}.{SCHEMA}.silver_player")
dim_player.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_player")
print(f"dim_player: {dim_player.count()} rows")

# COMMAND ----------
# --- dim_team ---
dim_team = spark.table(f"{CATALOG}.{SCHEMA}.silver_team")
dim_team.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_team")
print(f"dim_team: {dim_team.count()} rows")

# COMMAND ----------
# --- dim_league (join league + country) ---
league = spark.table(f"{CATALOG}.{SCHEMA}.silver_league")
country = spark.table(f"{CATALOG}.{SCHEMA}.silver_country")

dim_league = (
    league.join(country, on="country_id", how="left")
    .select("league_id", "league_name", "country_name")
)
dim_league.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_league")
print(f"dim_league: {dim_league.count()} rows")

# COMMAND ----------
# --- dim_player_ratings_history (SCD Type 2) ---
# silver_player_attributes has one row per player per date.
# We turn this into SCD2: each row is valid from its date until the
# next recorded date for that player (exclusive). The latest row per
# player is "current" (valid_to = NULL, is_current = true).

attrs = spark.table(f"{CATALOG}.{SCHEMA}.silver_player_attributes")

window_spec = Window.partitionBy("player_api_id").orderBy("date")

scd2 = (
    attrs
    .withColumn("valid_from", F.col("date"))
    .withColumn("next_date", F.lead("date").over(window_spec))
    # valid_to = day before the next recorded change, NULL if this is the latest
    .withColumn("valid_to", F.date_sub(F.col("next_date"), 1))
    .withColumn("is_current", F.col("next_date").isNull())
    .drop("date", "next_date")
)

dim_player_ratings_history = scd2.select(
    "player_api_id",
    "valid_from",
    "valid_to",
    "is_current",
    "overall_rating",
    "potential",
    "preferred_foot",
    "crossing",
    "finishing",
    "dribbling",
    "sprint_speed",
    "stamina"
)

dim_player_ratings_history.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.dim_player_ratings_history"
)
print(f"dim_player_ratings_history: {dim_player_ratings_history.count()} rows")

# COMMAND ----------
# --- fct_match_results ---
match = spark.table(f"{CATALOG}.{SCHEMA}.silver_match")
team = spark.table(f"{CATALOG}.{SCHEMA}.silver_team")

home_team = team.select(
    F.col("team_api_id").alias("home_team_api_id"),
    F.col("team_long_name").alias("home_team_name")
)
away_team = team.select(
    F.col("team_api_id").alias("away_team_api_id"),
    F.col("team_long_name").alias("away_team_name")
)

fct_match_results = (
    match
    .join(home_team, on="home_team_api_id", how="left")
    .join(away_team, on="away_team_api_id", how="left")
    .withColumn(
        "result",
        F.when(F.col("home_team_goal") > F.col("away_team_goal"), "HOME_WIN")
         .when(F.col("home_team_goal") < F.col("away_team_goal"), "AWAY_WIN")
         .otherwise("DRAW")
    )
    .withColumn("goal_difference", F.abs(F.col("home_team_goal") - F.col("away_team_goal")))
    .select(
        "match_api_id", "league_id", "season", "stage", "date",
        "home_team_api_id", "home_team_name",
        "away_team_api_id", "away_team_name",
        "home_team_goal", "away_team_goal",
        "result", "goal_difference"
    )
)

fct_match_results.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fct_match_results")
print(f"fct_match_results: {fct_match_results.count()} rows")

# COMMAND ----------
# Final check
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

# COMMAND ----------
# Sanity check: SCD2 example for one player
display(
    spark.table(f"{CATALOG}.{SCHEMA}.dim_player_ratings_history")
    .filter("player_api_id = 30981")
    .orderBy("valid_from")
)